# 05 – Content-based

**Content-based**: dùng đặc trưng nội dung phim, không cần rating của người khác → xử lý được phim mới.
- Đặc trưng phim: one-hot 18 thể loại + TF-IDF từ tiêu đề + thập niên phát hành. Mô hình không có siêu tham số cần dò.
- Hồ sơ user = tổng (rating − trung bình) × đặc trưng phim đã xem; điểm = cosine(hồ sơ, phim), quy về thang rating bằng $b_{ui}+w\cdot\cos$.

---
### Bản chất từng mô hình

**Content-based:** gợi ý dựa trên **nội dung phim**, không dùng rating của người khác.
- **Biểu diễn phim:** mỗi phim là một vector đặc trưng $\mathbf{f}_i$, đã chuẩn hóa L2.
  - One-hot 18 thể loại, cộng TF-IDF của các từ trong tiêu đề (từ hiếm như "Godfather" hay "Star" được trọng số cao hơn từ phổ biến) và thập niên phát hành.
- **Biểu diễn user (hồ sơ sở thích):**
$$\mathbf{u} = \sum_{i\,\text{đã chấm}} (r_{ui} - b_{ui})\,\mathbf{f}_i$$
  Phim được chấm cao hơn mức kỳ vọng kéo hồ sơ về phía đặc trưng của nó; phim bị chấm thấp hơn kỳ vọng đẩy hồ sơ ra xa.
- **Điểm:** $\cos(\mathbf{u}, \mathbf{f}_i)$ cho biết phim $i$ gần gu user đến đâu. Điểm này được quy về thang sao bằng $\hat r_{ui} = b_{ui} + w \cdot \cos$, với $w$ ước lượng bằng hồi quy tuyến tính trên train.

Điểm mạnh và điểm yếu:
- **Mạnh:**
  - Gợi ý được **phim mới chưa ai chấm** (cold-start phía phim), miễn là có thể loại và tiêu đề.
  - Dễ giải thích: "vì bạn thích phim hành động".
  - Không phụ thuộc user khác.
- **Yếu:**
  - Chỉ thấy những gì đặc trưng mô tả được: 18 thể loại quá thô để phân biệt phim hay với phim dở cùng thể loại.
  - Gợi ý bị "nhốt" trong vùng quen thuộc (filter bubble). Diversity thấp nhất trong 7 mô hình, khoảng 0.55.
  - User mới chưa chấm phim nào thì không có hồ sơ.

In [1]:
import sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))          # để import được package src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})

from src.config import RESULTS_DIR
from src.experiment import SELECT_METRIC, load_experiment, make_validation, run_model, save_metrics, validate

exp = load_experiment()                # cùng dữ liệu + cùng phép chia + cùng seed cho mọi notebook
FIG = RESULTS_DIR / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print(f"train={len(exp.train):,}  test={len(exp.test):,}  users={len(exp.user2idx):,}  items={len(exp.item2idx):,}")

train=800,193  test=200,016  users=6,040  items=3,706


In [2]:
from src.models.content_based import ContentBased

cb = ContentBased()
res = pd.DataFrame([run_model(cb, exp)])
print("done", cb.name)

done ContentBased(genres+title)


## Xem thử gợi ý content-based cho một user

In [3]:
uid = 1
print("Phim user", uid, "đã chấm cao (>=5):")
seen = exp.train[(exp.train.user_id == uid) & (exp.train.rating == 5)].merge(exp.movies, on="movie_id")
print(seen[["title_x", "genres_x"]].head(8).to_string(index=False, header=False))
print(f"\nGợi ý {cb.name}:")
cb.recommend(uid, k=10)[["title", "genres", "score"]]

Phim user 1 đã chấm cao (>=5):
     Back to the Future (1985)                Comedy|Sci-Fi
             Cinderella (1950) Animation|Children's|Musical
Last Days of Disco, The (1998)                        Drama
               Rain Man (1988)                        Drama
    Saving Private Ryan (1998)             Action|Drama|War
             Awakenings (1990)                        Drama
    Sound of Music, The (1965)                      Musical
              Apollo 13 (1995)                        Drama

Gợi ý ContentBased(genres+title):


,title,genres,score
0,"Shawshank Redemption, The (1994)",Drama,5.146906
1,To Kill a Mockingbird (1962),Drama,4.994001
2,Citizen Kane (1941),Drama,4.969085
3,Seven Samurai (The Magnificent Seven) (Shichin...,Action|Drama,4.953370
4,One Flew Over the Cuckoo's Nest (1975),Drama,4.953283
5,"Bridge on the River Kwai, The (1957)",Drama|War,4.949982
6,Paths of Glory (1957),Drama|War,4.937516
7,"Bicycle Thief, The (Ladri di biciclette) (1948)",Drama,4.918907
8,12 Angry Men (1957),Drama,4.884113
9,It's a Wonderful Life (1946),Drama,4.881513


In [4]:
res[['Model', 'Precision@10', 'Recall@10', 'NDCG@10', 'HitRate@10', 'Coverage@20', 'FitTime_s']].round(4)

,Model,Precision@10,Recall@10,NDCG@10,HitRate@10,Coverage@20,FitTime_s
0,ContentBased(genres+title),0.0715,0.0441,0.0875,0.4269,0.0982,0.9769


In [5]:
save_metrics(res.assign(Family="Content-based"), "content")

WindowsPath('D:/HỌC/kì 7/Machine Learning/machine_learning/outputs/metrics/content.csv')